In [1]:
import pandas as pd
import os

# Define the core paths (since the notebook is inside 'notebooks/' folder, we go one step back '..')
BASE_DIR = '../data/raw/CrisisMMD_v2.0/'
ANNOTATION_DIR = os.path.join(BASE_DIR, 'crisismmd_datasplit_all', 'crisismmd_datasplit_all')

# 1. Let's see what TSV files we have
print("Available Annotation Files:")
files = [f for f in os.listdir(ANNOTATION_DIR) if f.endswith('.tsv')]
for f in files:
    print(f" - {f}")

# 2. Load the primary training dataset annotation
# Task: Humanitarian (Informative vs Not Informative)
train_tsv_path = os.path.join(ANNOTATION_DIR, 'task_humanitarian_text_img_train.tsv')

if os.path.exists(train_tsv_path):
    df = pd.read_csv(train_tsv_path, sep='\t')
    print(f"\n[+] Total Data Points in Train Set: {len(df)}")
    print("\n[+] Columns in the dataset:")
    print(df.columns.tolist())
    
    # Display the first 3 rows to inspect the data structure
    display(df.head(3))
else:
    print(f"[-] File not found: {train_tsv_path}. Check the exact filenames printed above.")

Available Annotation Files:
 - task_damage_text_img_dev.tsv
 - task_damage_text_img_test.tsv
 - task_damage_text_img_train.tsv
 - task_humanitarian_text_img_dev.tsv
 - task_humanitarian_text_img_test.tsv
 - task_humanitarian_text_img_train.tsv
 - task_informative_text_img_dev.tsv
 - task_informative_text_img_test.tsv
 - task_informative_text_img_train.tsv

[+] Total Data Points in Train Set: 13608

[+] Columns in the dataset:
['event_name', 'tweet_id', 'image_id', 'tweet_text', 'image', 'label', 'label_text', 'label_image', 'label_text_image']


,event_name,tweet_id,image_id,tweet_text,image,label,label_text,label_image,label_text_image
0,california_wildfires,917791291823591425,917791291823591425_1,RT @Cal_OES: PLS SHARE: Weâ€™re capturing wild...,data_image/california_wildfires/10_10_2017/917...,not_humanitarian,other_relevant_information,not_humanitarian,Negative
1,california_wildfires,917791291823591425,917791291823591425_0,RT @Cal_OES: PLS SHARE: Weâ€™re capturing wild...,data_image/california_wildfires/10_10_2017/917...,other_relevant_information,other_relevant_information,infrastructure_and_utility_damage,Negative
2,california_wildfires,917793137925459968,917793137925459968_0,RT @KAKEnews: California wildfires destroy mor...,data_image/california_wildfires/10_10_2017/917...,infrastructure_and_utility_damage,infrastructure_and_utility_damage,infrastructure_and_utility_damage,Positive


In [2]:
# 1. Check the exact distribution of disaster events
print("--- Event Distribution (What kind of disasters are here?) ---")
display(df['event_name'].value_counts())

# 2. Check the distribution of image labels 
print("\n--- Image Label Distribution (What is actually in the images?) ---")
display(df['label_image'].value_counts())

--- Event Distribution (What kind of disasters are here?) ---


event_name
hurricane_maria         3453
hurricane_irma          3390
hurricane_harvey        3337
california_wildfires    1156
mexico_earthquake       1027
srilanka_floods          788
iraq_iran_earthquake     457
Name: count, dtype: int64


--- Image Label Distribution (What is actually in the images?) ---


label_image
not_humanitarian                          6565
infrastructure_and_utility_damage         2761
other_relevant_information                1860
rescue_volunteering_or_donation_effort    1694
affected_individuals                       397
vehicle_damage                             230
injured_or_dead_people                      92
missing_or_found_people                      9
Name: count, dtype: int64

In [3]:
# 1. Define the rigorous rules
allowed_events = ['hurricane_maria', 'hurricane_irma', 'hurricane_harvey', 'srilanka_floods']
discarded_labels = ['not_humanitarian', 'other_relevant_information']

# 2. Apply the filters to the dataframe
# Rule A: Keep only allowed water-related events
df_filtered = df[df['event_name'].isin(allowed_events)]

# Rule B: Drop completely useless labels
df_filtered = df_filtered[~df_filtered['label_image'].isin(discarded_labels)]

print(f"[+] Original Data Count: {len(df)}")
print(f"[+] Gold Standard Data Count After Filtration: {len(df_filtered)}")
print("\n[+] Remaining Valid Image Labels for DisasterNet-Bangla:")
display(df_filtered['label_image'].value_counts())

[+] Original Data Count: 13608
[+] Gold Standard Data Count After Filtration: 3755

[+] Remaining Valid Image Labels for DisasterNet-Bangla:


label_image
infrastructure_and_utility_damage         2079
rescue_volunteering_or_donation_effort    1185
affected_individuals                       306
vehicle_damage                             161
injured_or_dead_people                      21
missing_or_found_people                      3
Name: count, dtype: int64

In [4]:
import os
import shutil
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 1. Label Mapping Strategy (Macro-Classes)
label_mapping = {
    'infrastructure_and_utility_damage': 'Severe_Damage',
    'vehicle_damage': 'Severe_Damage',
    'rescue_volunteering_or_donation_effort': 'Humanitarian_Rescue',
    'affected_individuals': 'Affected_People',
    'injured_or_dead_people': 'Affected_People',
    'missing_or_found_people': 'Affected_People'
}

# Apply mapping to create a new column
df_filtered['macro_label'] = df_filtered['label_image'].map(label_mapping)

print("--- New Macro-Class Distribution ---")
display(df_filtered['macro_label'].value_counts())

# 2. Physical File Migration
RAW_DIR = '../data/raw/CrisisMMD_v2.0/'
PROCESSED_DIR = '../data/processed/'

print("\n[+] Starting physical file migration to data/processed/ ...")
successful_copies = 0
missing_files = 0

# Loop through the filtered dataframe and copy physical files
for index, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    # Original image path from TSV
    img_rel_path = row['image']
    src_path = os.path.join(RAW_DIR, img_rel_path)
    
    # Destination folder based on macro_label
    macro_label = row['macro_label']
    dest_folder = os.path.join(PROCESSED_DIR, macro_label)
    os.makedirs(dest_folder, exist_ok=True)
    
    # Extract filename and create dest path
    filename = os.path.basename(img_rel_path)
    dest_path = os.path.join(dest_folder, filename)
    
    # Copy file if source exists
    if os.path.exists(src_path):
        shutil.copy2(src_path, dest_path)
        successful_copies += 1
    else:
        missing_files += 1

print(f"\n[+] Migration Complete!")
print(f"[+] Successfully copied: {successful_copies} images")
if missing_files > 0:
    print(f"[-] Missing files (not found in raw directory): {missing_files}")

--- New Macro-Class Distribution ---


macro_label
Severe_Damage          2240
Humanitarian_Rescue    1185
Affected_People         330
Name: count, dtype: int64


[+] Starting physical file migration to data/processed/ ...


  0%|          | 0/3755 [00:00<?, ?it/s]

  1%|▏         | 48/3755 [00:00<00:07, 476.17it/s]

...

 97%|█████████▋| 3644/3755 [00:08<00:00, 490.33it/s]

 99%|█████████▊| 3699/3755 [00:08<00:00, 505.96it/s]

100%|█████████▉| 3752/3755 [00:08<00:00, 510.88it/s]

100%|██████████| 3755/3755 [00:08<00:00, 433.21it/s]


[+] Migration Complete!
[+] Successfully copied: 3755 images
